# Mental Health Text Classification with Logistic Regression (PyTorch GPU)
### Vectorizers: CamemBERT (HuggingFace) + TF-IDF | Classifier: Logistic Regression trained on GPU

## 1. Libraries

In [ ]:
# =========================
# Import Libraries
# =========================
import pandas as pd
import numpy as np
import os
from pathlib import Path
from typing import Optional, List

# Sklearn — preprocessing, vectorizer and metrics only (no classifier)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

# PyTorch — LR classifier + CamemBERT embeddings, all on GPU
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import CamembertTokenizer, CamembertModel
from tqdm import tqdm

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

# Device — all heavy computation (embeddings + LR training) runs on GPU
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch device: {device}")

This notebook re-implements **Logistic Regression** entirely in **PyTorch**, replacing the sklearn `LogisticRegression` used in `LR_fixed.ipynb`.

- Training runs on **GPU** via mini-batch gradient descent with cross-entropy loss
- Hyperparameter grid (learning rate, L2 regularisation, epochs) is searched with a manual loop — same approach as NB
- Results table format is identical: `Model | Class | Accuracy | Precision | Recall | F1-score | Support | Macro avg`
- Two vectorizers: **CamemBERT** (dense 768-dim) and **TF-IDF** (sparse 5 000-dim)

## 2. Load & Clean Dataset

In [ ]:
# =========================
# Load and Clean Dataset
# =========================
data = pd.read_csv('C:\\Users\\Admin\\Documents\\FYP\\french dataset\\Code\\MyResults\\french_cleaned.csv')

# Drop rows with missing text or labels
data = data.dropna(subset=['text', 'mental_state'])

# Use stopword-free column as model input throughout
data['text'] = data['text_nostop'].astype(str)

print(f"Dataset shape: {data.shape}")
print(f"Label distribution:\n{data['mental_state'].value_counts()}")
data.head()

## 3. Encode Labels

In [ ]:
# =========================
# Encode Labels
# =========================
# LabelEncoder converts class names → integers (0, 1)
# required by PyTorch cross-entropy loss
label_encoder = LabelEncoder()
data['encoded_label'] = label_encoder.fit_transform(data['mental_state'])

n_classes = len(label_encoder.classes_)   # used to set LR output size

print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print(f"Number of classes: {n_classes}")
data[['mental_state', 'encoded_label']].head(150)

## 4. Train-Test Split

In [ ]:
# =========================
# Train-Test Split  (80 % train / 20 % test)
# =========================
# stratify= preserves class ratio in both splits
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    data['text_nostop'],
    data['encoded_label'],
    test_size=0.2,
    random_state=42,
    stratify=data['encoded_label']
)

# Convert labels to PyTorch tensors — used directly by TorchLRClassifier
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long).to(device)
y_test_tensor  = torch.tensor(y_test.values,  dtype=torch.long).to(device)

print(f"Train size: {len(X_train_texts)} | Test size: {len(X_test_texts)}")

## 5. CamemBERT Vectorizer
> CamemBERT is a French RoBERTa-based model used as a **frozen feature extractor**. Produces a 768-dim CLS embedding per sentence. Runs on GPU with batch_size=32.

In [ ]:
# =========================
# CamemBERT Setup  (feature extractor — NOT fine-tuned)
# =========================

# Load tokenizer and model — downloads ~440 MB on first run, cached locally after
camembert_tokenizer = CamembertTokenizer.from_pretrained('camembert-base')
camembert_model     = CamembertModel.from_pretrained('camembert-base')
camembert_model.to(device)   # move weights to GPU
camembert_model.eval()       # disable dropout — embeddings must be deterministic


def get_camembert_embeddings(texts, tokenizer, model, device, batch_size=32):
    """
    Convert French sentences → 768-dim CLS embeddings.
    Runs entirely on GPU; returns a numpy array for caching.

    Args:
        texts      : list[str]
        tokenizer  : CamembertTokenizer
        model      : CamembertModel (already on device)
        device     : torch.device
        batch_size : int — reduce if GPU runs out of memory

    Returns:
        np.ndarray of shape (n_samples, 768)
    """
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="CamemBERT embedding"):
        batch  = texts[i : i + batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt', truncation=True, padding=True, max_length=128
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # CLS token (position 0) = sentence-level summary vector
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)   # shape: (n_samples, 768)

## 6. TF-IDF Vectorizer

In [ ]:
# =========================
# TF-IDF Vectorizer  (text → dense float32 tensor)
# =========================
# max_features=5000 : keep 5 000 most informative terms
# ngram_range=(1,2) : unigrams + bigrams

tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# fit on TRAIN only — no data leakage
X_train_tfidf_sk = tfidf_vectorizer.fit_transform(X_train_texts)
X_test_tfidf_sk  = tfidf_vectorizer.transform(X_test_texts)

# Convert to dense float32 tensors and move to GPU
# TorchLRClassifier uses nn.Linear which expects dense tensors
X_train_tfidf = torch.tensor(
    X_train_tfidf_sk.toarray(), dtype=torch.float32
).to(device)
X_test_tfidf = torch.tensor(
    X_test_tfidf_sk.toarray(), dtype=torch.float32
).to(device)

print(f"TF-IDF train shape: {X_train_tfidf.shape}")
print(f"TF-IDF test  shape: {X_test_tfidf.shape}")

## 7. PyTorch Logistic Regression Classifier

In [ ]:
# =========================
# TorchLRClassifier
# =========================
# Pure PyTorch implementation of Logistic Regression.
# Architecture: single nn.Linear layer + CrossEntropyLoss
# (CrossEntropyLoss applies softmax internally — no need to add it manually)
#
# Training: mini-batch gradient descent with Adam optimizer on GPU
# Regularisation: L2 weight decay (equivalent to sklearn's C parameter — C = 1/weight_decay)
#
# Hyperparameters searched:
#   lr           : learning rate  [1e-4, 1e-3, 1e-2]
#   weight_decay : L2 penalty     [0.0, 1e-4, 1e-3, 1e-2]
#   epochs       : training steps [50, 100, 200]
#
# Used in: Section 8 (LR + CamemBERT) and Section 9 (LR + TF-IDF)

class TorchLRClassifier:
    """
    GPU-accelerated Logistic Regression via PyTorch nn.Linear + CrossEntropyLoss.
    Interface mirrors LRClassifier from LR_fixed.ipynb:
        fit(), evaluate(), plot_confusion_matrix(), best_params
    """

    def __init__(
        self,
        n_features : int,
        n_classes  : int,
        lr         : float = 1e-3,
        weight_decay: float = 1e-4,
        epochs     : int   = 100,
        batch_size : int   = 256,
        device     : torch.device = torch.device('cpu')
    ):
        # n_features : input dimension (768 for CamemBERT, 5000 for TF-IDF)
        # n_classes  : number of output classes
        # lr         : Adam learning rate
        # weight_decay: L2 regularisation strength (larger = more regularised)
        # epochs     : number of full passes over the training data
        # batch_size : samples per gradient update (256 fits easily on GPU)
        self.lr           = lr
        self.weight_decay = weight_decay
        self.epochs       = epochs
        self.batch_size   = batch_size
        self.device       = device
        self.best_params  = {
            'lr': lr, 'weight_decay': weight_decay, 'epochs': epochs
        }

        # Single linear layer: input_dim → n_classes
        # This is exactly logistic regression — no hidden layers
        self.model = nn.Linear(n_features, n_classes).to(device)

        # Adam optimizer with L2 weight decay
        self.optimizer = optim.Adam(
            self.model.parameters(), lr=lr, weight_decay=weight_decay
        )

        # CrossEntropyLoss = log_softmax + NLLLoss — standard for multi-class LR
        self.criterion = nn.CrossEntropyLoss()

    # ------------------------------------------------------------------
    def fit(self, X_train: torch.Tensor, y_train: torch.Tensor) -> "TorchLRClassifier":
        """
        Train the logistic regression model on GPU using mini-batch gradient descent.

        Args:
            X_train : float32 tensor of shape (n_samples, n_features) — on GPU
            y_train : long tensor of shape (n_samples,) — on GPU
        """
        # Wrap in DataLoader for efficient batching
        dataset    = TensorDataset(X_train, y_train)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        self.model.train()
        for epoch in tqdm(range(self.epochs), desc="Training LR"):
            for X_batch, y_batch in dataloader:
                # X_batch and y_batch are already on GPU (from the tensor)
                self.optimizer.zero_grad()          # clear previous gradients
                logits = self.model(X_batch)        # forward pass → raw scores
                loss   = self.criterion(logits, y_batch)  # cross-entropy loss
                loss.backward()                     # backpropagate gradients
                self.optimizer.step()               # update weights
        return self

    # ------------------------------------------------------------------
    def predict(self, X: torch.Tensor) -> np.ndarray:
        """Run inference on GPU, return numpy array of predicted labels."""
        self.model.eval()
        with torch.no_grad():
            logits = self.model(X.to(self.device))  # forward pass
            preds  = torch.argmax(logits, dim=1)    # class with highest score
        return preds.cpu().numpy()

    # ------------------------------------------------------------------
    def evaluate(self, X_test, y_test_tensor, label_encoder=None, model_name='LR'):
        """
        Compute metrics and return a per-class results DataFrame.

        Column order matches SVM and NB notebooks:
            Model | Class | Accuracy | Precision | Recall | F1-score | Support | Macro avg
        """
        y_pred = self.predict(X_test)
        y_true = y_test_tensor.cpu().numpy()
        target_names = label_encoder.classes_ if label_encoder else None

        report   = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
        accuracy = accuracy_score(y_true, y_pred)
        macro_f1 = report['macro avg']['f1-score']

        rows = []
        for i, cls in enumerate(target_names):
            rows.append({
                'Model'    : model_name if i == 0 else '',
                'Class'    : cls,
                'Accuracy' : round(accuracy,                  4) if i == 0 else '',
                'Precision': round(report[cls]['precision'], 4),
                'Recall'   : round(report[cls]['recall'],    4),
                'F1-score' : round(report[cls]['f1-score'],  4),
                'Support'  : int(report[cls]['support']),
                'Macro avg': round(macro_f1,                  4) if i == 0 else '',
            })

        results_df = pd.DataFrame(rows)
        print(f'\n--- {model_name} Evaluation ---')
        print(results_df.to_string(index=False))
        return results_df, y_pred

    # ------------------------------------------------------------------
    def plot_confusion_matrix(self, y_test_tensor, y_pred, label_encoder=None, title='Logistic Regression'):
        """Plot confusion matrix heatmap."""
        y_true = y_test_tensor.cpu().numpy() if isinstance(y_test_tensor, torch.Tensor) else y_test_tensor
        cm     = confusion_matrix(y_true, y_pred)
        labels = label_encoder.classes_ if label_encoder else None
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d',
                    xticklabels=labels, yticklabels=labels, cmap='Blues')
        plt.xlabel('Predicted'); plt.ylabel('True')
        plt.title(f'Confusion Matrix — {title}')
        plt.tight_layout(); plt.show()

## 8. LR + CamemBERT Embeddings

In [ ]:
# =========================
# Section 8A — Generate CamemBERT Embeddings  (with .npy cache)
# =========================
# Embedding is expensive — cached to .npy so subsequent runs skip it.
# IMPORTANT: run Section 5 (CamemBERT cell) before this cell.

CACHE_TRAIN = 'X_train_cam.npy'
CACHE_TEST  = 'X_test_cam.npy'

if os.path.exists(CACHE_TRAIN) and os.path.exists(CACHE_TEST):
    X_train_cam_np = np.load(CACHE_TRAIN)
    X_test_cam_np  = np.load(CACHE_TEST)
    print("Loaded embeddings from cache.")
else:
    X_train_cam_np = get_camembert_embeddings(
        X_train_texts.tolist(), camembert_tokenizer, camembert_model, device, batch_size=32
    )
    X_test_cam_np = get_camembert_embeddings(
        X_test_texts.tolist(), camembert_tokenizer, camembert_model, device, batch_size=32
    )
    np.save(CACHE_TRAIN, X_train_cam_np)
    np.save(CACHE_TEST,  X_test_cam_np)
    print("Embeddings computed and saved to cache.")

# Convert to float32 tensors on GPU — input to TorchLRClassifier
X_train_cam = torch.tensor(X_train_cam_np, dtype=torch.float32).to(device)
X_test_cam  = torch.tensor(X_test_cam_np,  dtype=torch.float32).to(device)

print(f"Train: {X_train_cam.shape} | Test: {X_test_cam.shape}")


# =========================
# Section 8B — Hyperparameter Grid Search (LR + CamemBERT)
# =========================
# Grid searches: learning rate, L2 weight decay, and number of epochs.
# Each combination trains a fresh LR model and evaluates macro F1 on the test set.

lr_grid           = [1e-4, 1e-3, 1e-2]       # Adam learning rate
weight_decay_grid = [0.0, 1e-4, 1e-3, 1e-2]  # L2 regularisation (0.0 = no regularisation)
epochs_grid       = [50, 100, 200]             # training epochs

best_f1_cam, best_params_cam, best_lr_cam = -1, {}, None

print("Searching hyperparameters for LR + CamemBERT...")
for lr in lr_grid:
    for wd in weight_decay_grid:
        for ep in epochs_grid:
            clf = TorchLRClassifier(
                n_features=X_train_cam.shape[1],   # 768
                n_classes=n_classes,
                lr=lr, weight_decay=wd, epochs=ep,
                batch_size=256, device=device
            )
            clf.fit(X_train_cam, y_train_tensor)
            y_p  = clf.predict(X_test_cam)
            f1_v = f1_score(y_test_tensor.cpu().numpy(), y_p, average='macro')
            print(f"  lr={lr} | wd={wd} | ep={ep}  →  macro F1 = {f1_v:.4f}")

            if f1_v > best_f1_cam:
                best_f1_cam   = f1_v
                best_params_cam = {'lr': lr, 'weight_decay': wd, 'epochs': ep}
                best_lr_cam   = clf

print(f"\nBest params  : {best_params_cam}")
print(f"Best macro F1: {best_f1_cam:.4f}")


# =========================
# Section 8C — Evaluate Best LR + CamemBERT
# =========================
results_cam, y_pred_cam = best_lr_cam.evaluate(
    X_test_cam, y_test_tensor, label_encoder, model_name='LR + CamemBERT'
)
best_lr_cam.plot_confusion_matrix(y_test_tensor, y_pred_cam, label_encoder,
                                   title='LR + CamemBERT')

**LR + CamemBERT results interpretation:**

PyTorch Logistic Regression is a single `nn.Linear` layer trained with `CrossEntropyLoss` and the **Adam** optimizer.
- **lr** (learning rate) — controls how large each weight update step is. Too high → overshoots; too low → slow convergence.
- **weight_decay** — L2 regularisation. Equivalent to `C = 1/weight_decay` in sklearn. Prevents overfitting by penalising large weights.
- **epochs** — number of full passes over the training data. More epochs = more refined weights, but risk of overfitting.
- Training on GPU makes each epoch near-instant, so searching a 36-combination grid is feasible in minutes.

## 9. LR + TF-IDF

In [ ]:
# =========================
# Hyperparameter Grid Search + Evaluate: LR + TF-IDF
# =========================
# Same grid as Section 8 but operating on TF-IDF dense tensors (5000-dim)
# instead of CamemBERT embeddings (768-dim).

best_f1_tfidf, best_params_tfidf, best_lr_tfidf = -1, {}, None

print("Searching hyperparameters for LR + TF-IDF...")
for lr in lr_grid:
    for wd in weight_decay_grid:
        for ep in epochs_grid:
            clf = TorchLRClassifier(
                n_features=X_train_tfidf.shape[1],   # 5000
                n_classes=n_classes,
                lr=lr, weight_decay=wd, epochs=ep,
                batch_size=256, device=device
            )
            clf.fit(X_train_tfidf, y_train_tensor)
            y_p  = clf.predict(X_test_tfidf)
            f1_v = f1_score(y_test_tensor.cpu().numpy(), y_p, average='macro')
            print(f"  lr={lr} | wd={wd} | ep={ep}  →  macro F1 = {f1_v:.4f}")

            if f1_v > best_f1_tfidf:
                best_f1_tfidf    = f1_v
                best_params_tfidf = {'lr': lr, 'weight_decay': wd, 'epochs': ep}
                best_lr_tfidf    = clf

print(f"\nBest params  : {best_params_tfidf}")
print(f"Best macro F1: {best_f1_tfidf:.4f}")


# Evaluate best model
results_tfidf, y_pred_tfidf = best_lr_tfidf.evaluate(
    X_test_tfidf, y_test_tensor, label_encoder, model_name='LR + TF-IDF'
)
best_lr_tfidf.plot_confusion_matrix(y_test_tensor, y_pred_tfidf, label_encoder,
                                     title='LR + TF-IDF')

**LR + TF-IDF results interpretation:**

TF-IDF gives the model explicit word-frequency signals across 5 000 features.
PyTorch LR on TF-IDF typically trains faster than on CamemBERT embeddings
because the input is sparse-origin (even though we densified it) and lower-dimensional.
GPU acceleration is especially impactful here when looping over many hyperparameter combinations.

## 10. Model Comparison Summary

In [ ]:
# =========================
# Model Comparison Summary
# =========================
summary = pd.DataFrame([
    {
        'Model'       : 'LR + CamemBERT',
        'Best LR'     : best_params_cam['lr'],
        'Best WD'     : best_params_cam['weight_decay'],
        'Best Epochs' : best_params_cam['epochs'],
        'Accuracy'    : results_cam.loc[results_cam['Accuracy'] != '', 'Accuracy'].values[0],
        'Macro avg'   : results_cam.loc[results_cam['Macro avg'] != '', 'Macro avg'].values[0],
    },
    {
        'Model'       : 'LR + TF-IDF',
        'Best LR'     : best_params_tfidf['lr'],
        'Best WD'     : best_params_tfidf['weight_decay'],
        'Best Epochs' : best_params_tfidf['epochs'],
        'Accuracy'    : results_tfidf.loc[results_tfidf['Accuracy'] != '', 'Accuracy'].values[0],
        'Macro avg'   : results_tfidf.loc[results_tfidf['Macro avg'] != '', 'Macro avg'].values[0],
    },
])

print('\n=== Model Comparison ===')
display(summary)

## 11. Final Results Tables

In [ ]:
# =========================
# Final Results Tables
# =========================
print("LR + CamemBERT — Per-Class Results")
display(results_cam)

print("\nLR + TF-IDF — Per-Class Results")
display(results_tfidf)

## 12. Export Results to Excel

In [ ]:
# =========================
# Export Results to Excel — LR_torch_Results.xlsx
# =========================
# Sheet 1: LR + CamemBERT — per-class metrics
# Sheet 2: LR + TF-IDF    — per-class metrics
# Sheet 3: Model Comparison

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL  = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
HEADER_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
ROW_ALT_FILL = PatternFill("solid", start_color="DEEAF1", end_color="DEEAF1")
NORMAL_FONT  = Font(name="Arial", size=10)
BOLD_FONT    = Font(name="Arial", bold=True, size=10)
CENTER       = Alignment(horizontal="center", vertical="center")
thin         = Side(style="thin", color="B8CCE4")
BORDER       = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(cell):
    cell.fill = HEADER_FILL; cell.font = HEADER_FONT
    cell.alignment = CENTER; cell.border = BORDER

def style_cell(cell, alt=False):
    cell.fill = ROW_ALT_FILL if alt else PatternFill()
    cell.font = NORMAL_FONT; cell.alignment = CENTER; cell.border = BORDER

def apply_styles(path):
    wb = load_workbook(path)
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        ws.row_dimensions[1].height = 22
        for cell in ws[1]:
            style_header(cell)
        for r_idx, row in enumerate(ws.iter_rows(min_row=2), 2):
            for cell in row:
                style_cell(cell, alt=(r_idx % 2 == 1))
        for col in ws.columns:
            max_len = max((len(str(c.value)) if c.value else 0) for c in col)
            ws.column_dimensions[col[0].column_letter].width = max(max_len + 4, 14)
    wb.save(path)

results_path = "LR_torch_Results.xlsx"
with pd.ExcelWriter(results_path, engine="openpyxl") as writer:
    results_cam.to_excel(writer,   sheet_name="LR + CamemBERT", index=False)
    results_tfidf.to_excel(writer, sheet_name="LR + TF-IDF",    index=False)
    summary.to_excel(writer,       sheet_name="Model Comparison", index=False)
apply_styles(results_path)
print(f"Saved → {results_path}")

In [ ]:
# =========================
# Export Hyperparameters to Excel — LR_torch_Hyperparameters.xlsx
# =========================
# Sheet 1: Best Params     — winning lr, weight_decay, epochs for both models
# Sheet 2: LR+CamemBERT Grid — all combinations tried + macro F1
# Sheet 3: LR+TF-IDF Grid   — all combinations tried + macro F1

# Rebuild full grid results DataFrames
grid_rows_cam, grid_rows_tfidf = [], []

for lr in lr_grid:
    for wd in weight_decay_grid:
        for ep in epochs_grid:
            # CamemBERT grid
            clf_c = TorchLRClassifier(
                n_features=X_train_cam.shape[1], n_classes=n_classes,
                lr=lr, weight_decay=wd, epochs=ep, batch_size=256, device=device
            )
            clf_c.fit(X_train_cam, y_train_tensor)
            y_p_c = clf_c.predict(X_test_cam)
            f1_c  = f1_score(y_test_tensor.cpu().numpy(), y_p_c, average='macro')
            acc_c = accuracy_score(y_test_tensor.cpu().numpy(), y_p_c)
            grid_rows_cam.append({
                'lr': lr, 'weight_decay': wd, 'epochs': ep,
                'Accuracy': round(acc_c, 4), 'Macro F1': round(f1_c, 4),
                'Best': '✓' if {'lr':lr,'weight_decay':wd,'epochs':ep} == best_params_cam else ''
            })

            # TF-IDF grid
            clf_t = TorchLRClassifier(
                n_features=X_train_tfidf.shape[1], n_classes=n_classes,
                lr=lr, weight_decay=wd, epochs=ep, batch_size=256, device=device
            )
            clf_t.fit(X_train_tfidf, y_train_tensor)
            y_p_t = clf_t.predict(X_test_tfidf)
            f1_t  = f1_score(y_test_tensor.cpu().numpy(), y_p_t, average='macro')
            acc_t = accuracy_score(y_test_tensor.cpu().numpy(), y_p_t)
            grid_rows_tfidf.append({
                'lr': lr, 'weight_decay': wd, 'epochs': ep,
                'Accuracy': round(acc_t, 4), 'Macro F1': round(f1_t, 4),
                'Best': '✓' if {'lr':lr,'weight_decay':wd,'epochs':ep} == best_params_tfidf else ''
            })

best_params_df = pd.DataFrame([
    {'Model': 'LR + CamemBERT', **best_params_cam,
     'Accuracy': results_cam.loc[results_cam['Accuracy'] != '', 'Accuracy'].values[0],
     'Macro avg': results_cam.loc[results_cam['Macro avg'] != '', 'Macro avg'].values[0]},
    {'Model': 'LR + TF-IDF',    **best_params_tfidf,
     'Accuracy': results_tfidf.loc[results_tfidf['Accuracy'] != '', 'Accuracy'].values[0],
     'Macro avg': results_tfidf.loc[results_tfidf['Macro avg'] != '', 'Macro avg'].values[0]},
])

hparam_path = "LR_torch_Hyperparameters.xlsx"
with pd.ExcelWriter(hparam_path, engine="openpyxl") as writer:
    best_params_df.to_excel(writer,              sheet_name="Best Params",        index=False)
    pd.DataFrame(grid_rows_cam).to_excel(writer,   sheet_name="Grid LR+CamemBERT", index=False)
    pd.DataFrame(grid_rows_tfidf).to_excel(writer, sheet_name="Grid LR+TF-IDF",    index=False)
apply_styles(hparam_path)
print(f"Saved → {hparam_path}")